In [112]:
import pandas as pd


In [113]:
csv_path = "StarWars.csv"

In [114]:
df = pd.read_csv(csv_path, encoding="cp1252")
df.head()

,RespondentID,Have you seen any of the 6 films in the Star Wars franchise?,Do you consider yourself to be a fan of the Star Wars film franchise?,Which of the following Star Wars films have you seen? Please select all that apply.,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Please rank the Star Wars films in order of preference with 1 being your favorite film in the franchise and 6 being your least favorite film.,...,Unnamed: 28,Which character shot first?,Are you familiar with the Expanded Universe?,Do you consider yourself to be a fan of the Expanded Universe?Œæ,Do you consider yourself to be a fan of the Star Trek franchise?,Gender,Age,Household Income,Education,Location (Census Region)
0,NaN,Response,Response,Star Wars: Episode I The Phantom Menace,Star Wars: Episode II Attack of the Clones,Star Wars: Episode III Revenge of the Sith,Star Wars: Episode IV A New Hope,Star Wars: Episode V The Empire Strikes Back,Star Wars: Episode VI Return of the Jedi,Star Wars: Episode I The Phantom Menace,...,Yoda,Response,Response,Response,Response,Response,Response,Response,Response,Response
1,3.292880e+09,Yes,Yes,Star Wars: Episode I The Phantom Menace,Star Wars: Episode II Attack of the Clones,Star Wars: Episode III Revenge of the Sith,Star Wars: Episode IV A New Hope,Star Wars: Episode V The Empire Strikes Back,Star Wars: Episode VI Return of the Jedi,3,...,Very favorably,I don't understand this question,Yes,No,No,Male,18-29,NaN,High school degree,South Atlantic
2,3.292880e+09,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Yes,Male,18-29,"$0 - $24,999",Bachelor degree,West South Central
3,3.292765e+09,Yes,No,Star Wars: Episode I The Phantom Menace,Star Wars: Episode II Attack of the Clones,Star Wars: Episode III Revenge of the Sith,NaN,NaN,NaN,1,...,Unfamiliar (N/A),I don't understand this question,No,NaN,No,Male,18-29,"$0 - $24,999",High school degree,West North Central
4,3.292763e+09,Yes,Yes,Star Wars: Episode I The Phantom Menace,Star Wars: Episode II Attack of the Clones,Star Wars: Episode III Revenge of the Sith,Star Wars: Episode IV A New Hope,Star Wars: Episode V The Empire Strikes Back,Star Wars: Episode VI Return of the Jedi,5,...,Very favorably,I don't understand this question,No,NaN,Yes,Male,18-29,"$100,000 - $149,999",Some college or Associate degree,West North Central


The first row is a supplement for the headings and the formatting is horrendous so I decided to keep the row for reference in memory and call upon it when we need it to alter column names.

In [115]:
reference_row = df.iloc[0,:]
print(reference_row)
df.drop(index=0)
df = df.reset_index(drop=True)

RespondentID                                                                                                                                                                              NaN
Have you seen any of the 6 films in the Star Wars franchise?                                                                                                                         Response
Do you consider yourself to be a fan of the Star Wars film franchise?                                                                                                                Response
Which of the following Star Wars films have you seen? Please select all that apply.                                                                  Star Wars: Episode I  The Phantom Menace
Unnamed: 4                                                                                                                                        Star Wars: Episode II  Attack of the Clones
Unnamed: 5                                        

Here I am changing the the "Seen Movie X" column to a binary response rather than txt / NaN.
To do this, the notna() function will do exactly what we want, we just have to change the semantics of the column name.

In [116]:
movie_integer_counter = 1
new_columns = df.columns.tolist()
for col in range(3, 9):
    new_columns[col] = f"Has seen Star Wars Movie {movie_integer_counter}?"
    df.iloc[:, col] = df.iloc[:, col].notna()
    movie_integer_counter += 1

df.columns = new_columns
df = df.drop(index=0).reset_index(drop=True)

Similar approach here for the rankings, keeping 

In [117]:
movie_integer_counter = 1
new_columns = df.columns.tolist()
for col in range(9, 15):
    new_columns[col] = f"Ranking of Star Wars Movie{movie_integer_counter}?"
    df.iloc[:, col] = df.iloc[:, col].astype("Int64")
    movie_integer_counter += 1

df.columns = new_columns
df = df.drop(index=0).reset_index(drop=True)

In [118]:
favourability_ranking = df["Unnamed: 16"].value_counts().index.tolist()
favourability_ranking.remove("Unfamiliar (N/A)")
favourability_ranking

['Very favorably',
 'Somewhat favorably',
 'Neither favorably nor unfavorably (neutral)',
 'Somewhat unfavorably',
 'Very unfavorably']

In [119]:
from pandas.api.types import CategoricalDtype
import numpy as np

In [120]:
favourability_order = CategoricalDtype(categories=favourability_ranking, ordered = True)

In [121]:
df.iloc[:,15:29] = df.iloc[:,15:29].replace({"Unfamiliar (N/A)" : np.nan})
for i in range(15, 29):
    df.iloc[:, i] = df.iloc[:, i].astype(favourability_order)

In [122]:
df.head()

,RespondentID,Have you seen any of the 6 films in the Star Wars franchise?,Do you consider yourself to be a fan of the Star Wars film franchise?,Has seen Star Wars Movie 1?,Has seen Star Wars Movie 2?,Has seen Star Wars Movie 3?,Has seen Star Wars Movie 4?,Has seen Star Wars Movie 5?,Has seen Star Wars Movie 6?,Ranking of Star Wars Movie1?,...,Unnamed: 28,Which character shot first?,Are you familiar with the Expanded Universe?,Do you consider yourself to be a fan of the Expanded Universe?Œæ,Do you consider yourself to be a fan of the Star Trek franchise?,Gender,Age,Household Income,Education,Location (Census Region)
0,3.292880e+09,No,NaN,False,False,False,False,False,False,<NA>,...,NaN,NaN,NaN,NaN,Yes,Male,18-29,"$0 - $24,999",Bachelor degree,West South Central
1,3.292765e+09,Yes,No,True,True,True,False,False,False,1,...,NaN,I don't understand this question,No,NaN,No,Male,18-29,"$0 - $24,999",High school degree,West North Central
2,3.292763e+09,Yes,Yes,True,True,True,True,True,True,5,...,Very favorably,I don't understand this question,No,NaN,Yes,Male,18-29,"$100,000 - $149,999",Some college or Associate degree,West North Central
3,3.292731e+09,Yes,Yes,True,True,True,True,True,True,5,...,Somewhat favorably,Greedo,Yes,No,No,Male,18-29,"$100,000 - $149,999",Some college or Associate degree,West North Central
4,3.292719e+09,Yes,Yes,True,True,True,True,True,True,1,...,Very favorably,Han,Yes,No,Yes,Male,18-29,"$25,000 - $49,999",Bachelor degree,Middle Atlantic


In [123]:
new_columns = df.columns.tolist()
targets = new_columns[15:29]
counter = 15

# print(targets)

for col in targets:
        char = reference_row.iloc[counter]
        new_columns[counter] = f"Feelings on: {char}"
        counter += 1
        print(char)

df.columns = new_columns

df.head()
        

Han Solo
Luke Skywalker
Princess Leia Organa
Anakin Skywalker
Obi Wan Kenobi
Emperor Palpatine
Darth Vader
Lando Calrissian
Boba Fett
C-3P0
R2 D2
Jar Jar Binks
Padme Amidala
Yoda


,RespondentID,Have you seen any of the 6 films in the Star Wars franchise?,Do you consider yourself to be a fan of the Star Wars film franchise?,Has seen Star Wars Movie 1?,Has seen Star Wars Movie 2?,Has seen Star Wars Movie 3?,Has seen Star Wars Movie 4?,Has seen Star Wars Movie 5?,Has seen Star Wars Movie 6?,Ranking of Star Wars Movie1?,...,Feelings on: Yoda,Which character shot first?,Are you familiar with the Expanded Universe?,Do you consider yourself to be a fan of the Expanded Universe?Œæ,Do you consider yourself to be a fan of the Star Trek franchise?,Gender,Age,Household Income,Education,Location (Census Region)
0,3.292880e+09,No,NaN,False,False,False,False,False,False,<NA>,...,NaN,NaN,NaN,NaN,Yes,Male,18-29,"$0 - $24,999",Bachelor degree,West South Central
1,3.292765e+09,Yes,No,True,True,True,False,False,False,1,...,NaN,I don't understand this question,No,NaN,No,Male,18-29,"$0 - $24,999",High school degree,West North Central
2,3.292763e+09,Yes,Yes,True,True,True,True,True,True,5,...,Very favorably,I don't understand this question,No,NaN,Yes,Male,18-29,"$100,000 - $149,999",Some college or Associate degree,West North Central
3,3.292731e+09,Yes,Yes,True,True,True,True,True,True,5,...,Somewhat favorably,Greedo,Yes,No,No,Male,18-29,"$100,000 - $149,999",Some college or Associate degree,West North Central
4,3.292719e+09,Yes,Yes,True,True,True,True,True,True,1,...,Very favorably,Han,Yes,No,Yes,Male,18-29,"$25,000 - $49,999",Bachelor degree,Middle Atlantic


In [124]:
for i in range(30,33):
    df.iloc[:,i] = df.iloc[:, i].replace({"Yes" : True, "No" : False})

In [125]:
df.head()

,RespondentID,Have you seen any of the 6 films in the Star Wars franchise?,Do you consider yourself to be a fan of the Star Wars film franchise?,Has seen Star Wars Movie 1?,Has seen Star Wars Movie 2?,Has seen Star Wars Movie 3?,Has seen Star Wars Movie 4?,Has seen Star Wars Movie 5?,Has seen Star Wars Movie 6?,Ranking of Star Wars Movie1?,...,Feelings on: Yoda,Which character shot first?,Are you familiar with the Expanded Universe?,Do you consider yourself to be a fan of the Expanded Universe?Œæ,Do you consider yourself to be a fan of the Star Trek franchise?,Gender,Age,Household Income,Education,Location (Census Region)
0,3.292880e+09,No,NaN,False,False,False,False,False,False,<NA>,...,NaN,NaN,NaN,NaN,True,Male,18-29,"$0 - $24,999",Bachelor degree,West South Central
1,3.292765e+09,Yes,No,True,True,True,False,False,False,1,...,NaN,I don't understand this question,False,NaN,False,Male,18-29,"$0 - $24,999",High school degree,West North Central
2,3.292763e+09,Yes,Yes,True,True,True,True,True,True,5,...,Very favorably,I don't understand this question,False,NaN,True,Male,18-29,"$100,000 - $149,999",Some college or Associate degree,West North Central
3,3.292731e+09,Yes,Yes,True,True,True,True,True,True,5,...,Somewhat favorably,Greedo,True,False,False,Male,18-29,"$100,000 - $149,999",Some college or Associate degree,West North Central
4,3.292719e+09,Yes,Yes,True,True,True,True,True,True,1,...,Very favorably,Han,True,False,True,Male,18-29,"$25,000 - $49,999",Bachelor degree,Middle Atlantic
